# Econometría en Python

Cargar librerías necesarias:

In [110]:
import numpy as np
!pip install wooldridge
from wooldridge import *
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats


Cargamos datos:

In [ ]:
dataWoo("hprice3", description=True)

In [ ]:
datos = dataWoo("hprice3")
y = datos["price"]
X = datos[["area", "cbd", "baths", "y81"]]


## Estadísticos de los Datos

In [ ]:
import matplotlib.pyplot as plt

media=np.mean(y)
Q1=np.quantile(y, 0.25)
Q3=np.quantile(y, 0.75)
Varianza=np.var(y)
DesviacionTipica=np.std(y)
Mediana=np.median(y)
histograma=plt.hist(y, bins='auto', rwidth=0.85)
plt.xlabel('y')
plt.ylabel('Frecuencia')
plt.title("Histrograma de y (salary)")
plt.show()
print(Q1, Mediana, Q3, DesviacionTipica, np.mean(y))

Calculamos modelo de Mínimos Cuadrados Ordinarios

In [ ]:
mco = smf.ols("price ~ area + cbd + baths + y81", data=datos).fit()
mco.summary()

## Extracción de Información

In [ ]:
# Suma de Cuadrados Explicada (SCE)

SCE = mco.ess

#Suma de Cuadrados de los Residuos

SCR = mco.ssr

# Suma de Cuadrados Total

SCT = mco.centered_tss

print("SCR: ", SCR, "SCE: ", SCE, "SCT: ", SCT)

## Normalidad de los Residuos

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip
name = ["Jarque-Bera", "Chi^2 two-tail prob.", "Skew", "Kurtosis"]
test = sms.jarque_bera(mco.resid)
lzip(name, test)

## Multicolinealidad

In [ ]:
## Cálculo del FIV:
import statsmodels.stats.outliers_influence as oi

fivs=[oi.variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("FIVs: ", fivs)

### Cálculo del Número de condición

NC = np.sqrt(mco.condition_number)

### Matriz de Correlaciones:

corr_matrix=np.corrcoef(X.T)
print(corr_matrix)


In [ ]:
import statsmodels.graphics.api as smg
smg.plot_corr(corr_matrix, xnames=["area", "cbd", "baths", "y81"])
plt.show()

## Heteroscedasticidad

In [ ]:
#GOLDFELD-QUANDT (Muestras Pequeñas)
GQ=sms.het_goldfeldquandt(mco.model.endog, mco.model.exog, split=100)
print("GQ: ", GQ)

#BREUSH-PAGAN
BP=sms.het_breuschpagan(mco.resid, mco.model.exog)
print("BP: ", BP)

#WHITE
W=sms.het_white(mco.resid, mco.model.exog)

print("White: ", W)

In [ ]:
# Glejser:

z=datos["cbd"]
for h in [-2,-1,-0.5,0.5, 1, 2]:
    # |e| = delta_0 + delta_1 z^h + eps
    mcoaux=sm.OLS(abs(mco.resid), sm.add_constant(z**h)).fit()
    pval=mcoaux.pvalues["cbd"]
    print("h: ", h, "-> pvalt:", pval, "R2: ", mcoaux.rsquared)

#Mínimos Cuadrados Ponderados
mcp = sm.WLS(y, sm.add_constant(X), weights=1./np.sqrt(z**2)).fit()
mcp.summary()

## Autocorrelación

In [ ]:
### Durbin Watson:

from statsmodels.stats.stattools import durbin_watson
dw=durbin_watson(mco.resid)

print("DW: ", dw)


In [ ]:
## Corrección:

rho= 1 - dw/2

mco_autocorr=sm.GLSAR(y, sm.add_constant(X), rho=rho).iterative_fit(maxiter=1000)
mco_autocorr.summary()

# Cargar datos de ficheros:

In [108]:
#!pip install pandas
import pandas as pd #librería para manejo de datos

#### De CSV: Cargo mis datos
data= pd.read_csv('student-mat.csv', sep=";") #Lee parte de la base de datos "student-mat"
datos = pd.get_dummies(data)

datos.columns

Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2',
       'G3', 'school_GP', 'school_MS', 'sex_F', 'sex_M', 'address_R',
       'address_U', 'famsize_GT3', 'famsize_LE3', 'Pstatus_A', 'Pstatus_T',
       'Mjob_at_home', 'Mjob_health', 'Mjob_other', 'Mjob_services',
       'Mjob_teacher', 'Fjob_at_home', 'Fjob_health', 'Fjob_other',
       'Fjob_services', 'Fjob_teacher', 'reason_course', 'reason_home',
       'reason_other', 'reason_reputation', 'guardian_father',
       'guardian_mother', 'guardian_other', 'schoolsup_no', 'schoolsup_yes',
       'famsup_no', 'famsup_yes', 'paid_no', 'paid_yes', 'activities_no',
       'activities_yes', 'nursery_no', 'nursery_yes', 'higher_no',
       'higher_yes', 'internet_no', 'internet_yes', 'romantic_no',
       'romantic_yes'],
      dtype='object')

In [112]:
mco = smf.ols("G3 ~ G1 + G2 + internet_yes + Dalc + romantic_yes + health + sex_F + Mjob_at_home", data=datos).fit()
mco.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     G3   R-squared:                       0.824
Model:                            OLS   Adj. R-squared:                  0.821
Method:                 Least Squares   F-statistic:                     226.6
Date:                Tue, 12 Dec 2023   Prob (F-statistic):          1.02e-140
Time:                        19:48:35   Log-Likelihood:                -817.55
No. Observations:                 395   AIC:                             1653.
Df Residuals:                     386   BIC:                             1689.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -1.8887      0.578     -3.269      0.001      -3.025      -0.753
G1               0.1644      0.057      2.876      0.004       0.052       0.277
G2               0.9744      0.051     19.092      0.000       0.874       1.075
internet_yes    -0.0520      0.273     -0.190      0.849      -0.590       0.486
Dalc             0.0181      0.115      0.157      0.875      -0.208       0.244
romantic_yes    -0.3410      0.212     -1.607      0.109      -0.758       0.076
health           0.0769      0.072      1.067      0.287      -0.065       0.219
sex_F           -0.0909      0.209     -0.434      0.665      -0.503       0.321
Mjob_at_home    -0.1669      0.288     -0.580      0.562      -0.732       0.399
==============================================================================
Omnibus:                      232.368   Durbin-Watson:                   1.862
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1526.834
Skew:                          -2.535   Prob(JB):                         0.00
Kurtosis:                      11.189   Cond. No.                         102.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
import statsmodels.stats.outliers_influence as oi

fivs=[oi.variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("FIVs: ", fivs)

### Cálculo del Número de condición

NC = np.sqrt(mco.condition_number)

### Matriz de Correlaciones:

corr_matrix=np.corrcoef(X.T)
print(corr_matrix)